# Employee Attrition Prediction & HR Analytics

**Dataset:** IBM HR Analytics Employee Attrition Dataset
Source: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset

This notebook builds an end-to-end machine learning solution to analyze employee data,
identify the key factors driving employee attrition, and predict whether an employee
is likely to leave the company. It covers data cleaning, exploratory data analysis,
model training and comparison, hyperparameter tuning, and business recommendations.

## 1. Data Collection & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_PATH = "../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
df.describe()

### Data Cleaning

The dataset contains no missing values or duplicate rows. A few columns are constant
across all records (`EmployeeCount`, `Over18`, `StandardHours`) and provide no
predictive value, so they are dropped along with the `EmployeeNumber` identifier column.

In [ ]:
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print("Constant columns dropped:", constant_cols)

df_clean = df.drop(columns=constant_cols + ["EmployeeNumber"])
df_clean.to_csv("../data/processed/cleaned_attrition_data.csv", index=False)
df_clean.shape

## 2. Exploratory Data Analysis (EDA)

### 2.1 Overall Attrition Distribution

In [ ]:
attrition_rate = df_clean["Attrition"].value_counts(normalize=True) * 100
print(attrition_rate)

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df_clean, x="Attrition", hue="Attrition", legend=False, ax=ax)
ax.set_title("Overall Attrition Distribution")
plt.show()

**Insight:** The dataset is imbalanced -- roughly 84% of employees stayed while about
16% left the company. This imbalance is taken into account during model evaluation by
looking beyond accuracy at precision, recall, and F1-score.

### 2.2 Age Distribution by Attrition

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=df_clean, x="Age", hue="Attrition", kde=True, multiple="stack", ax=ax)
ax.set_title("Age Distribution by Attrition")
plt.show()

**Insight:** Younger employees, particularly those under 35, show a higher tendency to leave than older, more established employees.

### 2.3 Attrition by Department and Job Role

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df_clean, x="Department", hue="Attrition", ax=ax)
ax.set_title("Attrition by Department")
plt.xticks(rotation=15)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
order = df_clean["JobRole"].value_counts().index
sns.countplot(data=df_clean, y="JobRole", hue="Attrition", order=order, ax=ax)
ax.set_title("Attrition by Job Role")
plt.show()

**Insight:** Sales Representatives and Laboratory Technicians show noticeably higher attrition proportions relative to their department size than roles like Research Directors or Managers.

### 2.4 Attrition by OverTime and Salary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df_clean, x="OverTime", hue="Attrition", ax=ax)
ax.set_title("Attrition by OverTime Status")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df_clean, x="Attrition", y="MonthlyIncome", ax=ax)
ax.set_title("Monthly Income vs Attrition")
plt.show()

**Insight:** Employees who work overtime leave at a substantially higher rate than
those who don't. Employees who leave also tend to have a lower median monthly income,
suggesting compensation and workload both play a role in attrition.

### 2.5 Correlation Heatmap

In [ ]:
numeric_df = df_clean.select_dtypes(include=["int64", "float64"])
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap of Numeric Features")
plt.show()

**Insight:** `YearsAtCompany`, `YearsInCurrentRole`, `YearsWithCurrManager`, and
`TotalWorkingYears` are all strongly positively correlated with each other, indicating
overall employee tenure explains most of the variance in these related features.

### 2.6 Outlier Detection

In [ ]:
numeric_cols = ["MonthlyIncome", "TotalWorkingYears", "YearsAtCompany", "DistanceFromHome"]
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(16, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(data=df_clean, y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

**Insight:** `MonthlyIncome` and `TotalWorkingYears` show some high-end outliers corresponding to senior, long-tenured employees. These are legitimate data points rather than data errors, so they are retained for modeling.

## 3. Machine Learning Model

### 3.1 Encoding Categorical Variables

All categorical (text) columns are label-encoded so they can be used by the
classification algorithms. The target variable `Attrition` is mapped to a binary
0/1 label (`Yes` = 1, `No` = 0).

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

target = df_clean["Attrition"].map({"Yes": 1, "No": 0})
features = df_clean.drop(columns=["Attrition"])

categorical_cols = features.select_dtypes(include="object").columns.tolist()
print("Categorical columns:", categorical_cols)

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    features[col] = le.fit_transform(features[col])
    encoders[col] = le

features.head()

### 3.2 Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

### 3.3 Training Three Classification Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

def evaluate(name, model, X_te, y_te):
    preds = model.predict(X_te)
    print(f"--- {name} ---")
    print(classification_report(y_te, preds, target_names=["No", "Yes"]))
    return {
        "accuracy": accuracy_score(y_te, preds),
        "precision": precision_score(y_te, preds),
        "recall": recall_score(y_te, preds),
        "f1_score": f1_score(y_te, preds),
        "confusion_matrix": confusion_matrix(y_te, preds)
    }

results = {}

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
results["Logistic Regression"] = evaluate("Logistic Regression", log_reg, X_test_scaled, y_test)

In [ ]:
dtree = DecisionTreeClassifier(random_state=42)
dtree.fit(X_train, y_train)
results["Decision Tree"] = evaluate("Decision Tree", dtree, X_test, y_test)

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
results["Random Forest"] = evaluate("Random Forest", rf, X_test, y_test)

### 3.4 Model Comparison

In [ ]:
comparison_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k != "confusion_matrix"}
    for name, res in results.items()
}).T
comparison_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
comparison_df[["accuracy", "precision", "recall", "f1_score"]].plot(kind="bar", ax=ax)
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()

In [ ]:
for name, res in results.items():
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(res["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
                xticklabels=["No", "Yes"], yticklabels=["No", "Yes"], ax=ax)
    ax.set_title(f"Confusion Matrix - {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.show()

**Insight:** Logistic Regression achieves the strongest overall balance of precision,
recall, and F1-score for the minority (attrition = Yes) class, despite Random Forest
having a comparable accuracy. Because the business cares most about correctly
identifying employees who are likely to leave, F1-score on the "Yes" class is used
as the primary criterion for selecting the best model.

## 4. Model Improvement & Presentation

### 4.1 Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

best_model_name = max(results, key=lambda k: results[k]["f1_score"])
print("Best performing model (by F1-score):", best_model_name)

param_grid = {"C": [0.01, 0.1, 1, 10], "solver": ["lbfgs", "liblinear"]}
grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid, cv=5, scoring="f1", n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)
print("Best hyperparameters:", grid_search.best_params_)

tuned_model = grid_search.best_estimator_
tuned_results = evaluate("Logistic Regression (Tuned)", tuned_model, X_test_scaled, y_test)

### 4.2 Feature Importance

In [ ]:
importances = pd.Series(np.abs(tuned_model.coef_[0]), index=features.columns)
importances = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
importances.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top 15 Feature Importances (Logistic Regression Coefficients)")
plt.show()

**Insight:** `OverTime`, `YearsSinceLastPromotion`, `Department`, `NumCompaniesWorked`,
and `YearsWithCurrManager` are among the strongest predictors of attrition. This aligns with
the EDA findings that overtime and career stagnation are major drivers of turnover.

### 4.3 Business Recommendations

Based on the EDA and model results, the following actions are recommended to help
reduce employee attrition:

1. **Manage overtime carefully.** Employees who regularly work overtime leave at a much higher rate. Review workload distribution and staffing levels in high-overtime teams.
2. **Review compensation for at-risk roles.** Sales Representatives and Laboratory Technicians show higher attrition; benchmarking their pay against market rates could help retention.
3. **Support early-career and early-tenure employees.** Younger employees and those in their first few years show higher attrition; structured onboarding and mentorship programs may help.
4. **Expand stock option and long-term incentive eligibility.** Employees with low stock option levels are more likely to leave; broader eligibility could improve retention.
5. **Monitor job satisfaction and work-life balance scores.** These self-reported metrics are meaningfully associated with attrition and can serve as an early warning signal in regular employee surveys.

### 4.4 Conclusion

This project analyzed the IBM HR Analytics Employee Attrition dataset end-to-end --
from data cleaning and exploratory analysis through model training, comparison, and
tuning. The tuned Logistic Regression model provides a reasonable and interpretable
baseline for identifying employees at risk of attrition, and the feature importance
analysis highlights actionable levers (overtime, compensation, tenure, and stock
options) that HR can use to reduce turnover.